##Ingest drivers.json File
1.Read Data from circuits.csv using pyspark dataframe reader
2.Add Ingestion Metdata
    a.Add ingestion timestamp
    b.Add source
3.Write final dataframe to bronze schema

## Step1 .Read data 

In [0]:
%run ../00.common/01.environment_config

In [0]:
%run ../00.common/02.bronze_helpers

In [0]:
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,DateType
name_schema = StructType(
    [
        StructField('givenName',StringType(),True),
        StructField('familyName',StringType(),True)
    ]
)

drivers_schema = StructType(
    [
        StructField('driverId', StringType(), True),
        StructField('name', name_schema),
        StructField('dateOfBirth', DateType(), True),
        StructField('nationality', StringType(), True),
        StructField('url',StringType(),True)
    ]
)

In [0]:
# from pyspark.sql.types import StructType,StructField,StringType,FloatType

# circuits_schema = StructType(
#     [
#         StructField('circuitId', StringType(), True),
#         StructField('url', StringType(), True),
#         StructField('circuitName', StringType(), True),
#         StructField('lat', FloatType(), True),
#         StructField('long', FloatType(), True),
#         StructField('locality', StringType(), True),
#         StructField('country', StringType(), True)
#     ]
# )

In [0]:
drivers_df = (
    spark.read
    .format('json')
    .option('header',True)
    .schema(drivers_schema)
    .option('mode','FAILFAST')
    .load(source_file)
)

In [0]:
# display(drivers_df)

## Step2 . Add Ingestion Metadata

In [0]:
from pyspark.sql import functions as F
drivers_final_df = add_ingestion_metadata(drivers_df)

##Step3 .Write Data to Delta Table

In [0]:
(
    drivers_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql
-- SELECT * FROM formula1.bronze.circuits;

In [0]:
# display(spark.read.table(table_name))